## Problem 1: Euclidean vs. cosine on the unit sphere

**What we're proving:** when two vectors both have length 1, the squared distance between them depends only on the angle between them.

**Step 1: expand the distance.**

$$\|x - y\|_2^2 = \|x\|_2^2 - 2\langle x, y\rangle + \|y\|_2^2$$

This is just $(a - b)^2 = a^2 - 2ab + b^2$, written with vectors.

**Step 2: use length 1.**

Both lengths are 1, so $\|x\|_2^2 = \|y\|_2^2 = 1$.

Cosine similarity is $\cos\theta = \dfrac{\langle x, y\rangle}{\|x\|_2\,\|y\|_2}$. With lengths of 1 the denominator is 1, so $\cos\theta = \langle x, y\rangle$.

**Step 3: substitute.**

$$\|x - y\|_2^2 = 1 - 2\cos\theta + 1 = 2(1 - \cos\theta) \qquad \blacksquare$$

### Why the neighbour rankings are the same

From the formula, distance $= \sqrt{2(1 - \cos\theta)}$.

As $\cos\theta$ goes up, $1 - \cos\theta$ goes down, so the distance goes down. Higher cosine similarity always means smaller distance, and the relationship never flips (the function is strictly decreasing).

So ranking neighbours by "smallest Euclidean distance" or by "largest cosine similarity" gives exactly the same order and the same $k$ nearest neighbours. Only the scale is different.

| Vectors point... | $\cos\theta$ | Distance |
|---|---|---|
| the same way | 1 | 0 |
| at 90° | 0 | $\sqrt{2}$ |
| opposite ways | −1 | 2 |

## Problem 2: Cosine distance is not a metric

**Definition:** $d_{\cos}(x, y) = 1 - \cos\theta(x, y)$.

A metric needs four things:
1. $d(x, y) \ge 0$
2. $d(x, y) = 0$ only when $x = y$
3. $d(x, y) = d(y, x)$
4. **Triangle inequality:** $d(x, z) \le d(x, y) + d(y, z)$

Cosine distance passes the first three on the unit sphere, but it **fails the triangle inequality**.

### Counterexample in $\mathbb{R}^2$

Take three unit vectors at 0°, 45° and 90°:

$$x = (1, 0), \qquad y = \left(\tfrac{1}{\sqrt{2}}, \tfrac{1}{\sqrt{2}}\right), \qquad z = (0, 1)$$

Compute the cosines (all vectors have length 1, so cosine is just the dot product):

| Pair | Angle | $\cos\theta$ | $d_{\cos} = 1 - \cos\theta$ |
|---|---|---|---|
| $x, y$ | 45° | $\tfrac{\sqrt{2}}{2} \approx 0.707$ | $\approx 0.293$ |
| $y, z$ | 45° | $\tfrac{\sqrt{2}}{2} \approx 0.707$ | $\approx 0.293$ |
| $x, z$ | 90° | $0$ | $1$ |

Check the triangle inequality:

$$d_{\cos}(x, z) = 1 \quad > \quad d_{\cos}(x, y) + d_{\cos}(y, z) \approx 0.586$$

Going "through" $y$ is shorter than going straight from $x$ to $z$, which is impossible for a real distance. So $d_{\cos}$ is **not a metric**. $\blacksquare$

**Why it breaks:** from Problem 1, on the unit sphere $d_{\cos}(x, y) = \tfrac{1}{2}\|x - y\|_2^2$. That's a *squared* Euclidean distance, and squaring breaks the triangle inequality (e.g. on a line, $0 \to 1 \to 2$: $2^2 = 4 > 1^2 + 1^2 = 2$). For small angles, $1 - \cos\theta \approx \theta^2/2$, so small angles get shrunk too much compared to big ones.

### Contrast: angular distance

$$d_{\text{ang}}(x, y) = \frac{\arccos(\cos\theta)}{\pi} = \frac{\theta}{\pi} \in [0, 1]$$

This just measures the angle itself, scaled to $[0, 1]$. Same example:

| Pair | Angle | $d_{\text{ang}}$ |
|---|---|---|
| $x, y$ | 45° | $\tfrac{1}{4}$ |
| $y, z$ | 45° | $\tfrac{1}{4}$ |
| $x, z$ | 90° | $\tfrac{1}{2}$ |

$$d_{\text{ang}}(x, z) = \tfrac{1}{2} \quad \le \quad \tfrac{1}{4} + \tfrac{1}{4} = \tfrac{1}{2} \quad \checkmark$$

The triangle inequality holds (here with equality, since $y$ lies exactly between $x$ and $z$).

**Why angular distance is a metric:** the angle between two points on the unit sphere is the length of the shortest path along the sphere's surface (the great-circle arc). Shortest-path lengths always satisfy the triangle inequality, because a detour through $y$ can never beat the direct path. Dividing by $\pi$ just rescales it.

*Small note:* on vectors that aren't normalised, $x$ and $2x$ have angle 0, so $d_{\text{ang}}$ is only a metric on directions (i.e. on the unit sphere), not on raw vectors.

### Summary

| | $d_{\cos} = 1 - \cos\theta$ | $d_{\text{ang}} = \theta / \pi$ |
|---|---|---|
| Triangle inequality | ✗ fails | ✓ holds |
| Metric? | No | Yes (on the sphere) |
| Neighbour ranking | Same as angle | Same as angle |

Both are increasing functions of $\theta$, so they **rank neighbours the same way**. The difference only matters when you need actual metric properties, e.g. for metric-tree indexes (ball trees, VP-trees) or triangle-inequality pruning, which can give wrong results with $d_{\cos}$.

## Problem 3: Curse of dimensionality

### Part A: in high dimensions, all points are about equally far away

**Setup:** draw $n$ points $X^{(1)}, \dots, X^{(n)}$ uniformly from the cube $[0,1]^d$ and fix a query point $q$. Let $D_{\min}$ and $D_{\max}$ be the distances from $q$ to its nearest and farthest point. We want to show

$$\frac{D_{\max} - D_{\min}}{D_{\min}} \to 0 \quad \text{as } d \to \infty.$$

**Step 1: write the squared distance as a sum.**

For one random point $X$:

$$D^2 = \|X - q\|_2^2 = \sum_{j=1}^{d} \underbrace{(X_j - q_j)^2}_{Z_j}$$

The coordinates $X_j$ are independent, so $D^2$ is a sum of $d$ **independent** terms $Z_j$, each between 0 and 1.

**Step 2: the mean grows like $d$, the spread grows like $\sqrt{d}$.**

- Each $Z_j$ has mean at least $\tfrac{1}{12}$ (the smallest case, when $q_j = \tfrac12$). So the mean is $\mu = \mathbb{E}[D^2] \ge \tfrac{d}{12}$, which grows **linearly** in $d$.
- Each $Z_j$ lies in $[0, 1]$, so $\operatorname{Var}(Z_j) \le \tfrac14$. Independent variances add, so $\operatorname{Var}(D^2) \le \tfrac{d}{4}$, and the standard deviation is at most $\tfrac{\sqrt{d}}{2}$, which grows only like **$\sqrt{d}$**.

So the relative spread is

$$\frac{\text{std}(D^2)}{\mathbb{E}[D^2]} \le \frac{\sqrt{d}/2}{d/12} = \frac{6}{\sqrt{d}} \to 0.$$

The distance gets bigger, but its random wiggle gets **relatively** smaller. This is the law of large numbers.

**Step 3: Chebyshev's inequality.**

For any $\varepsilon > 0$ and one point:

$$P\big(|D^2 - \mu| \ge \varepsilon\mu\big) \le \frac{\operatorname{Var}(D^2)}{\varepsilon^2 \mu^2} \le \frac{d/4}{\varepsilon^2 d^2/144} = \frac{36}{\varepsilon^2 d}.$$

For all $n$ points:

$$P\big(\text{some point has } |D^2 - \mu| \ge \varepsilon\mu\big) \le \frac{36\,n}{\varepsilon^2 d} \to 0 \quad (n \text{ fixed}, \ d \to \infty).$$

So with probability going to 1, **every** squared distance is in $[(1-\varepsilon)\mu,\ (1+\varepsilon)\mu]$. Then

$$\frac{D_{\max}}{D_{\min}} \le \sqrt{\frac{1+\varepsilon}{1-\varepsilon}} \quad\Longrightarrow\quad \frac{D_{\max} - D_{\min}}{D_{\min}} \le \sqrt{\frac{1+\varepsilon}{1-\varepsilon}} - 1.$$

The right side can be made as small as we like by picking a small $\varepsilon$, so $\dfrac{D_{\max} - D_{\min}}{D_{\min}} \to 0$ in probability. $\blacksquare$

**Intuition:** $D$ grows like $\sqrt{d}$, but the gap $D_{\max} - D_{\min}$ stays roughly constant. A constant gap divided by something growing like $\sqrt{d}$ goes to 0, at a rate of about $1/\sqrt{d}$. The nearest and farthest points end up at almost the same distance.

### Part B: almost all of a high-dimensional ball is near its surface

**Step 1: volume scales like $r^d$.**

A $d$-dimensional ball of radius $r$ has volume $V_d(r) = C_d\, r^d$, where $C_d$ is a constant depending only on $d$.

**Step 2: compute the shell fraction.**

The shell of thickness $\varepsilon$ is the unit ball minus the inner ball of radius $1 - \varepsilon$:

$$\frac{V_d(1) - V_d(1-\varepsilon)}{V_d(1)} = \frac{C_d - C_d(1-\varepsilon)^d}{C_d} = 1 - (1-\varepsilon)^d.$$

**Step 3: take the limit.**

Since $0 < 1 - \varepsilon < 1$, raising it to a huge power sends it to 0. More precisely, $(1-\varepsilon)^d \le e^{-\varepsilon d} \to 0$. So

$$1 - (1-\varepsilon)^d \to 1. \qquad \blacksquare$$

**How fast?** Fraction of volume in the outer shell:

| $d$ | $\varepsilon = 0.01$ (outer 1%) | $\varepsilon = 0.1$ (outer 10%) |
|---|---|---|
| 2 | 2.0% | 19% |
| 10 | 9.6% | 65% |
| 100 | 63% | 99.997% |
| 1000 | 99.996% | ≈ 100% |

In 1000 dimensions, a skin only 1% thick holds essentially all the volume. The "inside" of the ball is basically empty.

### Consequences for nearest-neighbour search

- **"Nearest" loses meaning.** By Part A, the nearest point is barely closer than the farthest, so the neighbour ranking is unstable and small noise can change it.
- **Search gets slow.** Exact indexes (k-d trees, ball trees) rely on some points being clearly farther than others to skip them. When all distances are similar, they end up checking every point.
- **Data is spread thin.** By Part B, volume sits near the boundary and grows like $r^d$, so we need exponentially many points in $d$ to have truly close neighbours.

In practice NN search still works because real data has low *intrinsic* dimension, and we help it with dimensionality reduction (e.g. PCA, Problem 4) and approximate NN methods.

## Problem 4: PCA three ways

**Big picture.** PCA looks for the "best" directions to describe the data. "Best" can mean three things:
1. the direction where the data is **most spread out** (maximum variance),
2. the top **eigenvector of the covariance matrix**,
3. the directions that **lose the least information** when we project onto them (minimum reconstruction error).

We show all three give the same answer: the right singular vectors of $X$.

### Setup

- $X \in \mathbb{R}^{n \times p}$: $n$ data points (rows $x_i^\top$), $p$ features. It is **column-centred**: every column has mean 0.
- SVD: $X = U\Sigma V^\top$, where
  - $V = [v_1, \dots, v_p]$ is $p \times p$ orthogonal: its columns are orthonormal **directions in feature space**;
  - $\Sigma$ holds the singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$, which measure **how much the data stretches** along each $v_j$;
  - $U$ is $n \times n$ orthogonal, so $U^\top U = I$.

One identity we use everywhere:

$$X^\top X = V\Sigma^\top U^\top U \Sigma V^\top = V \Sigma^\top\Sigma V^\top = V\,\mathrm{diag}(\sigma_1^2, \dots, \sigma_p^2)\,V^\top$$

(with $\sigma_j = 0$ beyond the rank of $X$).

---

### Part 1: the max-variance direction is $v_1$

**What is $Xw$?** For a unit vector $w$, the $i$-th entry of $Xw$ is $x_i^\top w$, the coordinate of point $i$ along direction $w$. So $Xw$ is the data "squashed" onto the line through $w$.

**Its variance.** The columns of $X$ are centred, so $Xw$ also has mean 0. The variance is then just the average square:

$$\mathrm{Var}(Xw) = \frac{1}{n}\|Xw\|^2 = \frac{1}{n} w^\top X^\top X w.$$

**Rewrite in the $V$ basis.** Let $a = V^\top w$, the coordinates of $w$ in the basis $v_1, \dots, v_p$. Since $V$ is orthogonal, $\|a\| = \|w\| = 1$, i.e. $a_1^2 + \dots + a_p^2 = 1$. Then

$$w^\top X^\top X w = w^\top V\,\mathrm{diag}(\sigma_j^2)\,V^\top w = a^\top \mathrm{diag}(\sigma_j^2)\, a = \sum_{j=1}^{p} \sigma_j^2\, a_j^2.$$

**Maximise.** This is a weighted average of $\sigma_1^2, \dots, \sigma_p^2$ with weights $a_j^2$ that sum to 1. A weighted average can never exceed its largest value, so

$$\sum_j \sigma_j^2 a_j^2 \le \sigma_1^2 \sum_j a_j^2 = \sigma_1^2.$$

Equality holds when all the weight is on $j = 1$: $a = (1, 0, \dots, 0)$, i.e. $w = V a = v_1$. So

$$\boxed{\arg\max_{\|w\|=1} \mathrm{Var}(Xw) = v_1, \qquad \max \mathrm{Var} = \frac{\sigma_1^2}{n}.}$$

(Unique up to sign when $\sigma_1 > \sigma_2$.)

*Intuition:* any other direction is a mix of the $v_j$'s, and mixing in less-stretched directions only dilutes the spread.

---

### Part 2: $v_1$ is the top eigenvector of the covariance $\tfrac{1}{n}X^\top X$

Let $C = \frac{1}{n}X^\top X$ (the sample covariance, since $X$ is centred). From the identity above:

$$C = V\,\mathrm{diag}\!\left(\frac{\sigma_1^2}{n}, \dots, \frac{\sigma_p^2}{n}\right) V^\top.$$

Multiply by $v_1$. Since $V^\top v_1 = e_1$ (the first standard basis vector):

$$C v_1 = V\,\mathrm{diag}\!\left(\frac{\sigma_j^2}{n}\right) e_1 = \frac{\sigma_1^2}{n}\, V e_1 = \frac{\sigma_1^2}{n}\, v_1.$$

So $v_1$ is an eigenvector of $C$ with eigenvalue $\lambda_1 = \sigma_1^2/n$, and it is the **largest** eigenvalue because $\sigma_1$ is the largest singular value. In general:

$$\boxed{C v_j = \lambda_j v_j, \qquad \lambda_j = \frac{\sigma_j^2}{n}.}$$

**Same answer from the eigen side directly (Lagrange multipliers).** Maximise $w^\top C w$ subject to $w^\top w = 1$:

$$\nabla_w\big[w^\top C w - \lambda(w^\top w - 1)\big] = 2Cw - 2\lambda w = 0 \;\Longrightarrow\; Cw = \lambda w.$$

So any optimum is an eigenvector, and its objective value is $w^\top C w = \lambda w^\top w = \lambda$. The best choice is the eigenvector with the **largest** eigenvalue, which is $v_1$ again.

**Useful by-products (used in Problem 6):**
- the variance along $v_j$ is $\lambda_j = \sigma_j^2 / n$;
- the **explained-variance ratio** of component $j$ is $\dfrac{\lambda_j}{\sum_l \lambda_l} = \dfrac{\sigma_j^2}{\sum_l \sigma_l^2}$;
- scikit-learn divides by $n-1$ instead of $n$. That rescales every $\lambda_j$ by the same factor, so the directions and the ratios do not change.

---

### Part 3: the rank-$k$ PCA projection minimises reconstruction error

**What "rank-$k$ projection" means.** Pick $k$ orthonormal directions, stacked as the columns of $W \in \mathbb{R}^{p \times k}$ (so $W^\top W = I_k$). Project each point onto their span:

$$\hat{x}_i = W W^\top x_i.$$

$W^\top x_i$ gives the $k$ coordinates, and $W(\cdot)$ maps them back into $\mathbb{R}^p$. PCA uses $W = V_k = [v_1, \dots, v_k]$. We show this choice minimises

$$E(W) = \sum_{i=1}^{n} \|x_i - \hat{x}_i\|^2.$$

**Step 1: error = total − kept (Pythagoras).** The projection $\hat{x}_i$ and the residual $x_i - \hat{x}_i$ are perpendicular, so

$$\|x_i\|^2 = \|\hat{x}_i\|^2 + \|x_i - \hat{x}_i\|^2, \qquad \|\hat{x}_i\|^2 = \|W W^\top x_i\|^2 = \|W^\top x_i\|^2.$$

Summing over all points:

$$E(W) = \underbrace{\|X\|_F^2}_{\text{fixed}} - \underbrace{\|XW\|_F^2}_{\text{what we keep}}.$$

So **minimising the error is the same as maximising the kept spread** $\|XW\|_F^2 = \sum_{l=1}^k \|X w_l\|^2$, which is $n$ times the total variance along the $k$ directions. This is Part 1 again, with $k$ directions instead of one.

**Step 2: the most we can keep is $\sigma_1^2 + \dots + \sigma_k^2$.** Let $A = V^\top W$ (still $p \times k$ with orthonormal columns). As in Part 1:

$$\|XW\|_F^2 = \sum_{l=1}^k w_l^\top X^\top X w_l = \sum_{j=1}^{p} \sigma_j^2\, h_j, \qquad h_j = \sum_{l=1}^{k} A_{jl}^2 \ \ (\text{squared length of row } j \text{ of } A).$$

The weights $h_j$ have two properties:
- **They sum to $k$:** $\sum_j h_j = \|A\|_F^2 = k$, since $A$ has $k$ orthonormal columns, each of length 1.
- **Each is at most 1:** $h_j = (AA^\top)_{jj}$, and $AA^\top$ is an orthogonal projection $P$, so $h_j = e_j^\top P e_j = \|P e_j\|^2 \le \|e_j\|^2 = 1$.

So we have a total weight of $k$ to spread over $\sigma_1^2 \ge \sigma_2^2 \ge \dots$, with at most 1 per slot. The best we can do is fill the $k$ largest slots:

$$\|XW\|_F^2 \le \sigma_1^2 + \dots + \sigma_k^2.$$

$W = V_k$ achieves this ($h_1 = \dots = h_k = 1$, the rest 0).

**Conclusion.**

$$\boxed{\min_{W^\top W = I_k} \sum_i \|x_i - \hat{x}_i\|^2 = \|X\|_F^2 - \sum_{j=1}^{k} \sigma_j^2 = \sum_{j=k+1}^{p} \sigma_j^2, \quad \text{attained by } W = V_k.}$$

The error left over is exactly the squared singular values we threw away. (This is the Eckart–Young theorem.)

---

### Summary

| Question | Answer |
|---|---|
| Which unit direction has the most variance? | $v_1$, with variance $\sigma_1^2/n$ |
| Top eigenvector of the covariance $\frac{1}{n}X^\top X$? | $v_1$, with eigenvalue $\lambda_1 = \sigma_1^2/n$ |
| Best $k$-dim subspace for reconstruction? | $\mathrm{span}(v_1, \dots, v_k)$, error $= \sum_{j>k} \sigma_j^2$ |

All three come from one fact: $X^\top X = V\,\mathrm{diag}(\sigma_j^2)\,V^\top$. In practice we compute PCA with the **SVD of $X$**. It is numerically more stable than forming $X^\top X$, and it directly gives the directions ($V$), the variances ($\sigma_j^2/n$), and the projected coordinates $XV_k = U_k\Sigma_k$. That is what we implement in Problem 6.